# NYC WAV Service Risk: Data Ingestion and Preprocessing Pipeline

> **MAST30034 Applied Data Science · Project 1**  
**Part 1: TLC trip records and external weather/holiday preprocessing, target construction, and curated modelling panel generation.**

This notebook implements the end-to-end data ingestion and preprocessing pipeline. It ingests 2025 NYC TLC High-Volume For-Hire Vehicle (HVFHV) trip records, filters completed WAV requests, computes the signed arrival offset target metrics (`arrival_offset_minutes`, `late_10`), processes external NOAA weather observations and HRRR weather forecasts, builds a complete Borough × 4-Hour modelling grid, engineers historical rolling lag features, and materialises the curated modelling panel in `data/curated/`.

# 1. Project setup and configuration

In [51]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from tempfile import TemporaryDirectory

In [52]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

print(f"Project root: {PROJECT_ROOT}")
print(f"Python executable: {sys.executable}")

Project root: /Users/lavenderle/project-1-individual-LavenderLe7
Python executable: /Users/lavenderle/project-1-individual-LavenderLe7/.venv/bin/python


In [53]:
RANDOM_SEED = 42
TIMEZONE = "America/New_York"

ANALYSIS_YEAR = 2025
ANALYSIS_MONTHS = tuple(range(4, 13))

ANALYSIS_START = "2025-04-01 00:00:00"
TRAIN_END_EXCLUSIVE = "2025-09-01 00:00:00"
VALIDATION_END_EXCLUSIVE = "2025-11-01 00:00:00"
ANALYSIS_END_EXCLUSIVE = "2026-01-01 00:00:00"
TEST_END_EXCLUSIVE = ANALYSIS_END_EXCLUSIVE

PERIOD_HOURS = 4
FORECAST_HORIZON_PERIODS = 1
DATA_MATURITY_BUFFER_PERIODS = 1
HISTORICAL_FEATURE_LAG_PERIODS = (
    FORECAST_HORIZON_PERIODS + DATA_MATURITY_BUFFER_PERIODS
)
LATE_THRESHOLD_MINUTES = 10
WAV_REQUEST_FLAG_VALUE = "Y"

NYC_BOROUGHS = (
    "Bronx",
    "Brooklyn",
    "Manhattan",
    "Queens",
    "Staten Island",
)

HRRR_DATASET_ID = "noaa-hrrr-forecast-48-hour"
HRRR_INIT_START_UTC = "2025-03-31 00:00:00"
HRRR_INIT_END_UTC = "2026-01-01 06:00:00"
HRRR_INIT_FREQUENCY = "6h"
HRRR_FORECAST_AVAILABILITY_BUFFER_HOURS = 3
HRRR_DOWNLOAD_MAX_WORKERS = 8
HRRR_VARIABLES = (
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation_surface",
    "categorical_rain_surface",
    "wind_u_10m",
    "wind_v_10m",
    "wind_gust_surface",
)

In [54]:
# Input data roots
RAW_DATA_ROOT = PROJECT_ROOT / "data" / "raw"
REFERENCE_DATA_ROOT = PROJECT_ROOT / "data" / "reference"

# Pipeline-generated data roots
STANDARDIZED_DATA_ROOT = PROJECT_ROOT / "data" / "standardized"
CURATED_DATA_ROOT = PROJECT_ROOT / "data" / "curated"

In [55]:
# Processed data outputs
STANDARDIZED_WAV_REQUEST_ROOT = STANDARDIZED_DATA_ROOT / "wav_requests"
STANDARDIZED_WAV_REQUEST_PATHS = tuple(
    STANDARDIZED_WAV_REQUEST_ROOT
    / f"wav_requests_{ANALYSIS_YEAR}-{month:02d}.parquet"
    for month in ANALYSIS_MONTHS
)
HRRR_OUTPUT_ROOT = STANDARDIZED_DATA_ROOT / "hrrr_forecasts"
HRRR_COMBINED_PATH = (
    HRRR_OUTPUT_ROOT
    / "hrrr_borough_forecasts_standardized.parquet"
)
HRRR_GRID_MAPPING_PATH = (
    REFERENCE_DATA_ROOT / "hrrr_borough_grid_points.csv"
)
HRRR_TAXI_ZONE_SHAPEFILE_PATH = (
    RAW_DATA_ROOT / "tlc" / "taxi_zones" / "taxi_zones.shp"
)
HRRR_TAXI_ZONE_LOOKUP_PATH = (
    RAW_DATA_ROOT / "tlc" / "taxi_zone_lookup.csv"
)
CURATED_WAV_PANEL_PATH = (
    CURATED_DATA_ROOT
    / "wav_service_risk"
    / "borough_four_hour_panel.parquet"
)

# Report and model outputs
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
FIGURE_OUTPUT_ROOT = OUTPUT_ROOT / "figures"
TABLE_OUTPUT_ROOT = OUTPUT_ROOT / "tables"
MODEL_OUTPUT_ROOT = OUTPUT_ROOT / "model_results"

In [56]:
# Create required output directories
required_output_directories = (
    STANDARDIZED_WAV_REQUEST_ROOT,
    HRRR_OUTPUT_ROOT,
    HRRR_GRID_MAPPING_PATH.parent,
    CURATED_WAV_PANEL_PATH.parent,
    FIGURE_OUTPUT_ROOT,
    TABLE_OUTPUT_ROOT,
    MODEL_OUTPUT_ROOT,
)

for output_directory in required_output_directories:
    output_directory.mkdir(parents=True, exist_ok=True)

In [57]:
print(f"Analysis period: [{ANALYSIS_START}, {ANALYSIS_END_EXCLUSIVE})")
print(f"Training: [{ANALYSIS_START}, {TRAIN_END_EXCLUSIVE})")
print(
    f"Validation: [{TRAIN_END_EXCLUSIVE}, "
    f"{VALIDATION_END_EXCLUSIVE})"
)
print(
    f"Chronological evaluation: [{VALIDATION_END_EXCLUSIVE}, "
    f"{TEST_END_EXCLUSIVE})"
)
print(f"Forecast period: {PERIOD_HOURS} hours")
print(f"Late-arrival threshold: {LATE_THRESHOLD_MINUTES} minutes")
print(
    "HRRR forecast availability buffer: "
    f"{HRRR_FORECAST_AVAILABILITY_BUFFER_HOURS} hours"
)

Analysis period: [2025-04-01 00:00:00, 2026-01-01 00:00:00)
Training: [2025-04-01 00:00:00, 2025-09-01 00:00:00)
Validation: [2025-09-01 00:00:00, 2025-11-01 00:00:00)
Chronological evaluation: [2025-11-01 00:00:00, 2026-01-01 00:00:00)
Forecast period: 4 hours
Late-arrival threshold: 10 minutes
HRRR forecast availability buffer: 3 hours


# 2. Imports and Spark session

In [58]:
import certifi
import dynamical_catalog
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import seaborn as sns
import statsmodels.api as sm
import xarray as xr
from pyproj import CRS, Transformer
from pyspark import StorageLevel
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBRegressor

In [59]:
np.random.seed(RANDOM_SEED)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_theme(style="whitegrid", context="notebook")

In [60]:
SPARK_APP_NAME = "MAST30034_Assignment1"
SPARK_DRIVER_MEMORY = "4g"
SPARK_SHUFFLE_PARTITIONS = 16

# Initialise and configure the local Spark session
spark = (
    SparkSession.builder
    .appName(SPARK_APP_NAME)
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.memory", SPARK_DRIVER_MEMORY)
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.sql.shuffle.partitions", SPARK_SHUFFLE_PARTITIONS)
    .config("spark.sql.session.timeZone", TIMEZONE)
    .config("spark.sql.parquet.compression.codec", "zstd")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

In [61]:
print(
    f"Spark name: {spark.sparkContext.appName}, "
    f"Spark ready: {spark.sparkContext.master}, "
    f"timezone {spark.conf.get('spark.sql.session.timeZone')}"
)

Spark name: MAST30034_Assignment1, Spark ready: local[*], timezone America/New_York


# 3. TLC data preprocessing

## 3.1 HVFHV source data

Report the scale of the April–December source data and load only the fields needed for WAV trip preprocessing.

In [62]:
# Fixed HVFHV inputs for the study period
HVFHV_TRIP_ROOT = RAW_DATA_ROOT / "tlc"
EXPECTED_HVFHV_FILENAMES = tuple(
    f"fhvhv_tripdata_{ANALYSIS_YEAR}-{month:02d}.parquet"
    for month in ANALYSIS_MONTHS
)
EXPECTED_HVFHV_PATHS = tuple(
    HVFHV_TRIP_ROOT / file_name
    for file_name in EXPECTED_HVFHV_FILENAMES
)

# Fields required to define the WAV population and outcome
REQUIRED_HVFHV_COLUMNS = (
    "request_datetime",
    "on_scene_datetime",
    "pickup_datetime",
    "PULocationID",
    "wav_request_flag",
)

In [63]:
raw_row_count_from_metadata = sum(
    pq.ParquetFile(source_path).metadata.num_rows
    for source_path in EXPECTED_HVFHV_PATHS
)
raw_source_size_gb = (
    sum(source_path.stat().st_size for source_path in EXPECTED_HVFHV_PATHS)
    / 1024**3
)

print(
    f"Raw HVFHV source: {len(EXPECTED_HVFHV_PATHS)} monthly files, "
    f"{raw_row_count_from_metadata:,} rows, "
    f"{raw_source_size_gb:.2f} GiB"
)

Raw HVFHV source: 9 monthly files, 183,307,678 rows, 4.20 GiB


`request_datetime` anchors the four-hour target period, while `on_scene_datetime` defines vehicle arrival. `pickup_datetime` confirms a completed trip, `PULocationID` supplies pickup geography, and `wav_request_flag` identifies WAV-requested trips.

In [64]:
raw_hvfhv_df = (
    spark.read
    .parquet(*[str(path) for path in EXPECTED_HVFHV_PATHS])
    .select(*REQUIRED_HVFHV_COLUMNS)
)

## 3.2 TLC pickup geography

Join the TLC Taxi Zone lookup so each trip can be assigned to one of the five pickup boroughs, and prepare the corresponding geometries for later mapping.

In [65]:
TAXI_ZONE_LOOKUP_PATH = RAW_DATA_ROOT / "tlc" / "taxi_zone_lookup.csv"

TAXI_ZONE_LOOKUP_SCHEMA = T.StructType(
    [
        T.StructField("LocationID", T.IntegerType(), False),
        T.StructField("Borough", T.StringType(), True),
        T.StructField("Zone", T.StringType(), True),
        T.StructField("service_zone", T.StringType(), True),
    ]
)

taxi_zone_lookup_df = (
    spark.read
    .option("header", True)
    .schema(TAXI_ZONE_LOOKUP_SCHEMA)
    .csv(str(TAXI_ZONE_LOOKUP_PATH))
    .select(
        F.col("LocationID").alias("pickup_location_id"),
        F.col("Zone").alias("pickup_zone"),
        F.col("Borough").alias("pickup_borough"),
    )
)

In [66]:
raw_hvfhv_profile_df = (
    raw_hvfhv_df
    .join(
        F.broadcast(taxi_zone_lookup_df),
        F.col("PULocationID") == F.col("pickup_location_id"),
        how="left",
    )
    .drop("pickup_location_id")
    .withColumn(
        "valid_nyc_pickup_zone",
        F.col("pickup_borough").isin(*NYC_BOROUGHS),
    )
)

### Taxi Zone geometry for mapping

Combine the TLC Taxi Zone polygons with lookup attributes to create the map-ready geography.

In [67]:
TAXI_ZONE_SHAPEFILE_PATH = (
    RAW_DATA_ROOT
    / "tlc"
    / "taxi_zones"
    / "taxi_zones.shp"
)

# Load the official Taxi Zone shapes and lookup attributes
taxi_zone_lookup_pandas_df = pd.read_csv(TAXI_ZONE_LOOKUP_PATH)
taxi_zone_shapes_gdf = gpd.read_file(TAXI_ZONE_SHAPEFILE_PATH)

# Replace abbreviated shapefile labels with TLC lookup names
taxi_zones_gdf = (
    taxi_zone_shapes_gdf
    .drop(columns=["zone", "borough"])
    .merge(
        taxi_zone_lookup_pandas_df,
        on="LocationID",
        how="left",
    )
    .rename(
        columns={
            "LocationID": "pickup_location_id",
            "Zone": "pickup_zone",
            "Borough": "pickup_borough",
        }
    )
)

print(f"Taxi Zone geometries: {len(taxi_zones_gdf):,}")
print(f"Taxi Zone CRS: {taxi_zones_gdf.crs}")

Taxi Zone geometries: 263
Taxi Zone CRS: EPSG:2263


## 3.3 WAV analysis population

In [68]:
# Identify WAV-requested trips
is_wav_requested = (
    F.upper(F.trim(F.col("wav_request_flag")))
    == WAV_REQUEST_FLAG_VALUE
)

# Require complete timestamps for completed-trip outcome construction
has_completed_trip_timestamps = (
    F.col("request_datetime").isNotNull()
    & F.col("on_scene_datetime").isNotNull()
    & F.col("pickup_datetime").isNotNull()
)

# Restrict trips to the fixed analysis interval
is_in_analysis_interval = (
    (F.col("request_datetime") >= F.lit(ANALYSIS_START))
    & (F.col("request_datetime") < F.lit(ANALYSIS_END_EXCLUSIVE))
)

# Require a valid NYC pickup zone
has_valid_nyc_pickup = F.coalesce(
    F.col("valid_nyc_pickup_zone"),
    F.lit(False),
)

# Combine eligibility conditions for the WAV analysis population
valid_wav_population = (
    is_wav_requested
    & has_completed_trip_timestamps
    & is_in_analysis_interval
    & has_valid_nyc_pickup
)

In [69]:
# Define cumulative filtering stages for the WAV analysis population
filter_stage_definitions = (
    ("Raw HVFHV records", F.lit(True)),
    ("Passenger requested a WAV", is_wav_requested),
    (
        "Required trip timestamps present",
        is_wav_requested & has_completed_trip_timestamps,
    ),
    (
        "Requested pickup inside study interval",
        is_wav_requested
        & has_completed_trip_timestamps
        & is_in_analysis_interval,
    ),
    ("Valid five-borough pickup zone", valid_wav_population),
)

# Count records retained at all filtering stages in a single Spark aggregation
filter_count_record = (
    raw_hvfhv_profile_df
    .agg(
        *(
            F.sum(condition.cast("long")).alias(f"stage_{index}")
            for index, (_, condition) in enumerate(
                filter_stage_definitions
            )
        )
    )
    .first()
    .asDict()
)

# Summarise stage-specific exclusions and retention relative to the raw data
filtering_summary_records = []
raw_filter_row_count = filter_count_record["stage_0"]
previous_row_count = raw_filter_row_count

for stage_order, (stage_name, _) in enumerate(filter_stage_definitions):
    stage_row_count = filter_count_record[f"stage_{stage_order}"]
    excluded_row_count = previous_row_count - stage_row_count
    filtering_summary_records.append(
        {
            "stage": stage_name,
            "row_count": stage_row_count,
            "excluded_at_stage": excluded_row_count,
            "retained_percentage_of_raw": (
                100 * stage_row_count / raw_filter_row_count
            ),
        }
    )
    previous_row_count = stage_row_count

wav_filtering_summary_df = pd.DataFrame(filtering_summary_records)
display(wav_filtering_summary_df)

wav_population_df = raw_hvfhv_profile_df.filter(valid_wav_population)

,stage,row_count,excluded_at_stage,retained_percentage_of_raw
0,Raw HVFHV records,183307678,0,100.0000
1,Passenger requested a WAV,543349,182764329,0.2964
2,Required trip timestamps present,543349,0,0.2964
3,Requested pickup inside study interval,543347,2,0.2964
4,Valid five-borough pickup zone,543305,42,0.2964


## 3.4 Arrival-offset target construction and validation

In [70]:
# Calculate vehicle arrival offset from the requested pickup time
arrival_offset_minutes = (
    F.timestamp_diff(
        "SECOND",
        "request_datetime",
        "on_scene_datetime",
    )
    / F.lit(60.0)
)

# Assign each request to the start of its fixed-length time period
period_start_hour = (
    F.floor(
        F.hour("request_datetime") / F.lit(PERIOD_HOURS)
    )
    * F.lit(PERIOD_HOURS)
).cast("int")
period_start = F.make_timestamp_ntz(
    years=F.year("request_datetime"),
    months=F.month("request_datetime"),
    days=F.dayofmonth("request_datetime"),
    hours=period_start_hour,
    mins=F.lit(0),
    secs=F.lit(0),
)

In [71]:
# Construct trip-level outcome and temporal variables for WAV analysis
wav_target_df = (
    wav_population_df
    .withColumn("arrival_offset_minutes", arrival_offset_minutes)
    .withColumn(
        "late_10",
        (F.col("arrival_offset_minutes") > F.lit(LATE_THRESHOLD_MINUTES)).cast("int"),
    )
    .withColumn("requested_pickup_date", F.to_date("request_datetime"))
    .withColumn("period_start", period_start)
    .withColumn("year", F.year("request_datetime"))
    .withColumn("month", F.month("request_datetime"))
    .persist(StorageLevel.MEMORY_AND_DISK)
)

In [72]:
# Restrict outcome diagnostics to the April-October development period
development_wav_target_df = wav_target_df.filter(
    F.col("request_datetime") < F.lit(VALIDATION_END_EXCLUSIVE)
)

# Summarise development-period completion delays
development_completion_delay = (
    development_wav_target_df
    .agg(
        F.max(
            F.timestamp_diff(
                "SECOND",
                "request_datetime",
                "on_scene_datetime",
            ) / F.lit(60.0)
        ).alias("maximum_on_scene_delay_minutes"),
        F.max(
            F.timestamp_diff(
                "SECOND",
                "request_datetime",
                "pickup_datetime",
            ) / F.lit(60.0)
        ).alias("maximum_pickup_delay_minutes"),
    )
    .first()
    .asDict()
)
# Calculate development-period vehicle arrival-offset statistics
offset_summary = (
    development_wav_target_df
    .agg(
        F.count("*").alias("record_count"),
        F.min("arrival_offset_minutes").alias("minimum_minutes"),
        F.max("arrival_offset_minutes").alias("maximum_minutes"),
        F.sum(
            (F.col("arrival_offset_minutes") < 0).cast("long")
        ).alias("negative_count"),
        F.sum("late_10").alias("late_10_count"),
    )
    .first()
    .asDict()
)

# Calculate selected quantiles of the arrival-offset distribution
OFFSET_QUANTILE_PROBABILITIES = (0.01, 0.50, 0.95, 0.99)
OFFSET_QUANTILE_NAMES = (
    "p01_minutes",
    "median_minutes",
    "p95_minutes",
    "p99_minutes",
)
offset_quantiles = development_wav_target_df.approxQuantile(
    "arrival_offset_minutes",
    list(OFFSET_QUANTILE_PROBABILITIES),
    0.0001,
)

# Combine counts, quantiles, and percentages into one summary table
offset_profile_record = {
    **offset_summary,
    **development_completion_delay,
    **dict(zip(OFFSET_QUANTILE_NAMES, offset_quantiles)),
    "negative_percentage": (
        100
        * offset_summary["negative_count"]
        / offset_summary["record_count"]
    ),
    "late_10_percentage": (
        100
        * offset_summary["late_10_count"]
        / offset_summary["record_count"]
    ),
}

development_arrival_offset_profile_df = pd.DataFrame(
    [offset_profile_record]
)
development_arrival_offset_profile_df.T.rename(
    columns={0: "value"}
)

,value
record_count,428633.0000
minimum_minutes,-51.5333
maximum_minutes,212.4000
negative_count,82927.0000
late_10_count,40374.0000
maximum_on_scene_delay_minutes,212.4000
maximum_pickup_delay_minutes,212.4000
p01_minutes,-15.2333
median_minutes,4.3000
p95_minutes,12.1833


## 3.5 Standardised WAV trip output

In [73]:
STANDARDIZED_WAV_SCHEMA = {
    "requested_pickup_datetime": "timestamp_ntz",
    "on_scene_datetime": "timestamp_ntz",
    "pickup_datetime": "timestamp_ntz",
    "pickup_location_id": "int",
    "pickup_zone": "string",
    "pickup_borough": "string",
    "arrival_offset_minutes": "double",
    "late_10": "int",
    "period_start": "timestamp_ntz",
    "month": "int",
}

In [74]:
# Create and save the standardized trip-level WAV dataset
wav_standardized_df = wav_target_df.select(
    F.col("request_datetime").alias("requested_pickup_datetime"),
    "on_scene_datetime",
    "pickup_datetime",
    F.col("PULocationID").alias("pickup_location_id"),
    "pickup_zone",
    "pickup_borough",
    "arrival_offset_minutes",
    "late_10",
    "period_start",
    "month",
)

for month, output_path in zip(
    ANALYSIS_MONTHS,
    STANDARDIZED_WAV_REQUEST_PATHS,
):
    with TemporaryDirectory(
        prefix=f".wav_requests_{ANALYSIS_YEAR}_{month:02d}_",
        dir=STANDARDIZED_WAV_REQUEST_ROOT,
    ) as temporary_directory:
        (
            wav_standardized_df
            .filter(F.col("month") == month)
            .coalesce(1)
            .write
            .mode("overwrite")
            .parquet(temporary_directory)
        )
        temporary_part_path = next(
            Path(temporary_directory).glob("part-*.parquet")
        )
        temporary_part_path.replace(output_path)

wav_target_df.unpersist()

print(
    f"Saved standardised WAV trips to "
    f"{STANDARDIZED_WAV_REQUEST_ROOT}"
)

Saved standardised WAV trips to /Users/lavenderle/project-1-individual-LavenderLe7/data/standardized/wav_requests


# 4. External data preprocessing

Prepare observed Central Park weather for descriptive analysis, archived
borough-point HRRR forecasts for issue-time-eligible model features, and
the known-in-advance federal-holiday reference.

## 4.1 NOAA observed weather

Parse Central Park LCD observations, align fixed standard time with New
York civil time, and aggregate them for development-period weather EDA.
These realised observations are descriptive context, not model predictors.

In [75]:
# Define NOAA source settings and required weather fields
NOAA_FIXED_STANDARD_TIMEZONE = "Etc/GMT+5"
NOAA_HOURLY_REPORT_TYPES = ("FM-15", "FM-16")
NOAA_WEATHER_COLUMNS = (
    "STATION",
    "DATE",
    "REPORT_TYPE",
    "HourlyDryBulbTemperature",
    "HourlyPrecipitation",
    "HourlyWindSpeed",
    "HourlyVisibility",
    "HourlyPresentWeatherType",
)

In [76]:
NOAA_STATION_ID = "USW00094728"
NOAA_WEATHER_PATH = (
    RAW_DATA_ROOT
    / "external"
    / "noaa_lcd"
    / f"LCD_{NOAA_STATION_ID}_{ANALYSIS_YEAR}.csv"
)

weather_raw_df = pd.read_csv(
    NOAA_WEATHER_PATH,
    usecols=NOAA_WEATHER_COLUMNS,
    dtype="string",
)
weather_raw_df = weather_raw_df.loc[
    weather_raw_df["REPORT_TYPE"].isin(NOAA_HOURLY_REPORT_TYPES)
].copy()

# Convert NOAA observation times to New York local time
weather_source_datetime = pd.to_datetime(
    weather_raw_df["DATE"],
    errors="coerce",
)
weather_raw_df["observation_datetime"] = (
    weather_source_datetime
    .dt.tz_localize(NOAA_FIXED_STANDARD_TIMEZONE)
    .dt.tz_convert(TIMEZONE)
    .dt.tz_localize(None)
)

In [77]:
# Convert fields stored as standard numeric strings
weather_raw_df["temperature_c"] = pd.to_numeric(
    weather_raw_df["HourlyDryBulbTemperature"],
    errors="coerce",
)
weather_raw_df["wind_speed_ms"] = pd.to_numeric(
    weather_raw_df["HourlyWindSpeed"],
    errors="coerce",
)

# Extract the numeric component from the few flagged visibility values
visibility_values = weather_raw_df[
    "HourlyVisibility"
].str.extract(r"([-+]?\d+(?:\.\d+)?)", expand=False)
weather_raw_df["visibility_km"] = pd.to_numeric(
    visibility_values,
    errors="coerce",
)

In [78]:
# Preserve trace precipitation and leave unreported amounts missing
precipitation_values = (
    weather_raw_df["HourlyPrecipitation"]
    .str.strip()
    .str.upper()
)
weather_raw_df["precipitation_mm"] = pd.to_numeric(
    precipitation_values,
    errors="coerce",
)
weather_raw_df["is_trace_precipitation"] = (
    precipitation_values.eq("T")
)

# Standardise the composite NOAA weather-type codes
weather_codes = (
    weather_raw_df["HourlyPresentWeatherType"]
    .fillna("")
    .str.upper()
)

# Identify rain and snow conditions
weather_raw_df["is_rain"] = weather_codes.str.contains(
    r"(?:RA|DZ)",
    regex=True,
)
weather_raw_df["is_snow"] = weather_codes.str.contains(
    r"(?:SN|SG|PL|IC)",
    regex=True,
)
weather_raw_df["has_precipitation"] = (
    weather_raw_df["precipitation_mm"].gt(0)
    | weather_raw_df["is_trace_precipitation"]
    | weather_codes.str.contains(
        r"(?:RA|DZ|SN|SG|PL|IC|UP|GR|GS)",
        regex=True,
    )
).fillna(False)

In [79]:
# Aggregate NOAA observations to hourly weather conditions
weather_raw_df["observation_hour"] = (
    weather_raw_df["observation_datetime"].dt.floor("h")
)

weather_hourly_df = (
    weather_raw_df
    .groupby("observation_hour", as_index=True)
    .agg(
        temperature_c=("temperature_c", "median"),
        wind_speed_ms=("wind_speed_ms", "max"),
        visibility_km=("visibility_km", "min"),
        has_precipitation=("has_precipitation", "max"),
        is_rain=("is_rain", "max"),
        is_snow=("is_snow", "max"),
        observation_count=("REPORT_TYPE", "size"),
    )
)

# Reindex to the full study-period hourly grid and flag observed hours
expected_weather_hours = pd.date_range(
    start=ANALYSIS_START,
    end=ANALYSIS_END_EXCLUSIVE,
    inclusive="left",
    freq="h",
)

weather_hourly_df = weather_hourly_df.reindex(expected_weather_hours)
weather_hourly_df.index.name = "observation_hour"
weather_hourly_df["weather_observed"] = (
    weather_hourly_df["observation_count"].fillna(0).gt(0)
)

In [80]:
# Aggregate hourly conditions to the four-hour modelling periods
weather_hourly_for_period_df = weather_hourly_df.reset_index()
weather_hourly_for_period_df["period_start"] = (
    weather_hourly_for_period_df["observation_hour"].dt.floor("4h")
)
weather_period_df = (
    weather_hourly_for_period_df
    .groupby("period_start", as_index=False)
    .agg(
        observed_mean_temperature_c=("temperature_c", "mean"),
        observed_max_wind_speed_ms=("wind_speed_ms", "max"),
        observed_min_visibility_km=("visibility_km", "min"),
        observed_any_precipitation=("has_precipitation", "max"),
        observed_any_rain=("is_rain", "max"),
        observed_any_snow=("is_snow", "max"),
        weather_observation_count=("weather_observed", "sum"),
    )
)
weather_period_df["weather_coverage_share"] = (
    weather_period_df["weather_observation_count"] / PERIOD_HOURS
)

# Summarise source coverage before joining weather to the panel
weather_quality_summary_df = pd.DataFrame(
    [
        {
            "expected_hour_count": len(expected_weather_hours),
            "observed_hour_count": int(
                weather_hourly_df["weather_observed"].sum()
            ),
            "hourly_coverage_percentage": (
                100 * weather_hourly_df["weather_observed"].mean()
            ),
            "complete_period_count": int(
                weather_period_df["weather_coverage_share"].eq(1).sum()
            ),
            "period_count": len(weather_period_df),
        }
    ]
)
weather_quality_summary_df

,expected_hour_count,observed_hour_count,hourly_coverage_percentage,complete_period_count,period_count
0,6600,6483,98.2273,1606,1650


In [81]:
# Fix the schema for a stable pandas-to-Spark conversion
WEATHER_PERIOD_SCHEMA = T.StructType(
    [
        T.StructField("period_start", T.TimestampNTZType(), False),
        T.StructField("observed_mean_temperature_c", T.DoubleType()),
        T.StructField("observed_max_wind_speed_ms", T.DoubleType()),
        T.StructField("observed_min_visibility_km", T.DoubleType()),
        T.StructField("observed_any_precipitation", T.BooleanType()),
        T.StructField("observed_any_rain", T.BooleanType()),
        T.StructField("observed_any_snow", T.BooleanType()),
        T.StructField("weather_observation_count", T.LongType()),
        T.StructField("weather_coverage_share", T.DoubleType()),
    ]
)
# Convert pandas and NumPy values to Spark-compatible Python values
weather_period_records = []
for weather_record in weather_period_df.to_dict(orient="records"):
    cleaned_weather_record = {
        field_name: (
            None
            if pd.isna(field_value)
            else (
                field_value.item()
                if isinstance(field_value, np.generic)
                else field_value
            )
        )
        for field_name, field_value in weather_record.items()
    }
    cleaned_weather_record["period_start"] = weather_record[
        "period_start"
    ].to_pydatetime()
    weather_period_records.append(cleaned_weather_record)

weather_period_spark_df = spark.createDataFrame(
    weather_period_records,
    schema=WEATHER_PERIOD_SCHEMA,
)

## 4.2 Archived HRRR forecast acquisition and standardisation

Materialise restartable monthly subsets of the HRRR archive for one
representative grid point per borough. Retain complete forecast runs here;
issue-time eligibility is imposed later in the forecast-design section.

In [82]:
# Use the project environment's trusted CA bundle on macOS builds
os.environ["SSL_CERT_FILE"] = certifi.where()

hrrr_requested_init_times = pd.date_range(
    start=HRRR_INIT_START_UTC,
    end=HRRR_INIT_END_UTC,
    freq=HRRR_INIT_FREQUENCY,
)

pd.DataFrame(
    [
        {
            "dataset_id": HRRR_DATASET_ID,
            "first_init_time_utc": hrrr_requested_init_times.min(),
            "last_init_time_utc": hrrr_requested_init_times.max(),
            "borough_count": len(NYC_BOROUGHS),
            "weather_variable_count": len(HRRR_VARIABLES),
            "output_root": str(HRRR_OUTPUT_ROOT),
        }
    ]
)

,dataset_id,first_init_time_utc,last_init_time_utc,borough_count,weather_variable_count,output_root
0,noaa-hrrr-forecast-48-hour,2025-03-31,2026-01-01 06:00:00,5,7,/Users/lavenderle/project-1-individual-Lavende...


In [83]:
# Build one representative point per borough from TLC Taxi Zone shapes
hrrr_taxi_zone_shapes_gdf = gpd.read_file(
    HRRR_TAXI_ZONE_SHAPEFILE_PATH
)
hrrr_taxi_zone_lookup_df = pd.read_csv(HRRR_TAXI_ZONE_LOOKUP_PATH)
hrrr_taxi_zones_gdf = (
    hrrr_taxi_zone_shapes_gdf
    .drop(columns=["zone", "borough"])
    .merge(
        hrrr_taxi_zone_lookup_df,
        on="LocationID",
        how="left",
    )
)
hrrr_borough_geometry_gdf = (
    hrrr_taxi_zones_gdf.loc[
        hrrr_taxi_zones_gdf["Borough"].isin(NYC_BOROUGHS)
    ]
    .dissolve(by="Borough")
    .reindex(NYC_BOROUGHS)
)
hrrr_borough_centroids_gdf = hrrr_borough_geometry_gdf.copy()
hrrr_borough_centroids_gdf["geometry"] = (
    hrrr_borough_geometry_gdf.geometry.centroid
)
hrrr_borough_centroids_wgs84_gdf = (
    hrrr_borough_centroids_gdf.to_crs("EPSG:4326")
)

# Open metadata lazily; no full-CONUS forecast array is loaded here
hrrr_remote_ds = dynamical_catalog.open(
    HRRR_DATASET_ID,
    chunks=None,
)
hrrr_projected_crs = CRS.from_wkt(
    hrrr_remote_ds.spatial_ref.attrs["crs_wkt"]
)
hrrr_coordinate_transformer = Transformer.from_crs(
    "EPSG:4326",
    hrrr_projected_crs,
    always_xy=True,
)
hrrr_projected_x, hrrr_projected_y = (
    hrrr_coordinate_transformer.transform(
        hrrr_borough_centroids_wgs84_gdf.geometry.x.to_numpy(),
        hrrr_borough_centroids_wgs84_gdf.geometry.y.to_numpy(),
    )
)
hrrr_x_indices = np.abs(
    hrrr_remote_ds.x.values[:, None] - hrrr_projected_x[None, :]
).argmin(axis=0)
hrrr_y_indices = np.abs(
    hrrr_remote_ds.y.values[:, None] - hrrr_projected_y[None, :]
).argmin(axis=0)
hrrr_x_indexer = xr.DataArray(
    hrrr_x_indices,
    dims="borough",
    coords={"borough": list(NYC_BOROUGHS)},
)
hrrr_y_indexer = xr.DataArray(
    hrrr_y_indices,
    dims="borough",
    coords={"borough": list(NYC_BOROUGHS)},
)

# Map requested initialization times to archive positions
hrrr_remote_init_time_index = pd.Index(
    pd.to_datetime(hrrr_remote_ds.init_time.values)
)
hrrr_init_position_by_time = {
    init_time: int(hrrr_remote_init_time_index.get_loc(init_time))
    for init_time in hrrr_requested_init_times
}
hrrr_selected_grid_latitude = hrrr_remote_ds.latitude.isel(
    x=hrrr_x_indexer,
    y=hrrr_y_indexer,
).values
hrrr_selected_grid_longitude = hrrr_remote_ds.longitude.isel(
    x=hrrr_x_indexer,
    y=hrrr_y_indexer,
).values
hrrr_grid_mapping_df = pd.DataFrame(
    {
        "borough": NYC_BOROUGHS,
        "borough_centroid_latitude": (
            hrrr_borough_centroids_wgs84_gdf.geometry.y.to_numpy()
        ),
        "borough_centroid_longitude": (
            hrrr_borough_centroids_wgs84_gdf.geometry.x.to_numpy()
        ),
        "hrrr_grid_latitude": hrrr_selected_grid_latitude,
        "hrrr_grid_longitude": hrrr_selected_grid_longitude,
        "hrrr_x_index": hrrr_x_indices,
        "hrrr_y_index": hrrr_y_indices,
        "hrrr_x_m": hrrr_remote_ds.x.values[hrrr_x_indices],
        "hrrr_y_m": hrrr_remote_ds.y.values[hrrr_y_indices],
    }
)
hrrr_grid_mapping_df.to_csv(HRRR_GRID_MAPPING_PATH, index=False)
hrrr_grid_mapping_df

,borough,borough_centroid_latitude,borough_centroid_longitude,hrrr_grid_latitude,hrrr_grid_longitude,hrrr_x_index,hrrr_y_index,hrrr_x_m,hrrr_y_m
0,Bronx,40.8526,-73.8665,40.8530,-73.8694,1555,357,1967479.8575,515693.8474
1,Brooklyn,40.6455,-73.9498,40.6444,-73.9416,1555,365,1967479.8575,491693.8474
2,Manhattan,40.7776,-73.9671,40.7884,-73.9654,1553,360,1961479.8575,506693.8474
3,Queens,40.7055,-73.8192,40.7021,-73.8113,1558,362,1976479.8575,500693.8474
4,Staten Island,40.5808,-74.1534,40.5740,-74.1492,1550,369,1952479.8575,479693.8474


In [84]:
def load_hrrr_forecast_run(init_time: pd.Timestamp) -> pd.DataFrame:
    """Load one HRRR run for the five frozen borough grid points.

    Args:
        init_time: Naive UTC model-initialization timestamp.

    Returns:
        Long-form hourly forecast rows for every lead and borough.
    """
    init_position = hrrr_init_position_by_time[init_time]
    selected_run_ds = (
        hrrr_remote_ds[list(HRRR_VARIABLES)]
        .isel(
            init_time=[init_position],
            x=hrrr_x_indexer,
            y=hrrr_y_indexer,
        )
        .load()
    )

    run_df = selected_run_ds.to_dataframe().reset_index()
    run_df["init_time_utc"] = pd.to_datetime(
        run_df["init_time"],
        utc=True,
    )
    run_df["valid_time_utc"] = pd.to_datetime(
        run_df["valid_time"],
        utc=True,
    )
    run_df["lead_time_hours"] = (
        run_df["lead_time"].dt.total_seconds() / 3600
    ).astype("int16")
    run_df["precipitation_rate_mm_per_hour"] = (
        3600 * run_df["precipitation_surface"]
    )
    run_df["categorical_rain"] = (
        run_df["categorical_rain_surface"]
        .round()
        .astype("Int8")
    )
    run_df["wind_speed_10m_ms"] = np.hypot(
        run_df["wind_u_10m"],
        run_df["wind_v_10m"],
    )
    run_df = run_df.rename(
        columns={
            "latitude": "grid_latitude",
            "longitude": "grid_longitude",
            "temperature_2m": "temperature_2m_c",
            "relative_humidity_2m": (
                "relative_humidity_2m_pct"
            ),
            "wind_u_10m": "wind_u_10m_ms",
            "wind_v_10m": "wind_v_10m_ms",
            "wind_gust_surface": "wind_gust_ms",
        }
    )
    run_df["source_dataset"] = HRRR_DATASET_ID
    output_columns = (
        "source_dataset",
        "init_time_utc",
        "valid_time_utc",
        "lead_time_hours",
        "borough",
        "grid_latitude",
        "grid_longitude",
        "temperature_2m_c",
        "relative_humidity_2m_pct",
        "precipitation_rate_mm_per_hour",
        "categorical_rain",
        "wind_u_10m_ms",
        "wind_v_10m_ms",
        "wind_speed_10m_ms",
        "wind_gust_ms",
    )
    return run_df.loc[:, output_columns]


def download_hrrr_month(
    init_times: pd.DatetimeIndex,
    output_path: Path,
) -> pd.DataFrame:
    """Load a cached monthly HRRR partition or download it once."""
    if output_path.exists():
        print(f"Loading cached partition: {output_path.name}")
        return pd.read_parquet(output_path)

    run_frames = []
    with ThreadPoolExecutor(
        max_workers=HRRR_DOWNLOAD_MAX_WORKERS
    ) as executor:
        for completed_count, run_df in enumerate(
            executor.map(load_hrrr_forecast_run, init_times),
            start=1,
        ):
            run_frames.append(run_df)
            if (
                completed_count % 16 == 0
                or completed_count == len(init_times)
            ):
                print(
                    f"{output_path.stem}: "
                    f"{completed_count}/{len(init_times)} runs",
                    flush=True,
                )

    monthly_df = (
        pd.concat(run_frames, ignore_index=True)
        .sort_values(
            ["init_time_utc", "lead_time_hours", "borough"]
        )
        .reset_index(drop=True)
    )
    monthly_df.to_parquet(output_path, index=False)
    return monthly_df

In [85]:
# Load cached monthly partitions or download missing months
hrrr_monthly_frames = []
hrrr_requested_months = hrrr_requested_init_times.to_period("M").unique()
for requested_month in hrrr_requested_months:
    monthly_init_times = hrrr_requested_init_times[
        hrrr_requested_init_times.to_period("M") == requested_month
    ]
    monthly_output_path = (
        HRRR_OUTPUT_ROOT
        / f"hrrr_forecast_init_{requested_month}.parquet"
    )
    hrrr_monthly_frames.append(
        download_hrrr_month(monthly_init_times, monthly_output_path)
    )

hrrr_forecast_df = (
    pd.concat(hrrr_monthly_frames, ignore_index=True)
    .sort_values(
        ["init_time_utc", "lead_time_hours", "borough"]
    )
    .reset_index(drop=True)
)
hrrr_duplicate_key_count = int(
    hrrr_forecast_df.duplicated(
        ["init_time_utc", "lead_time_hours", "borough"]
    ).sum()
)
hrrr_precipitation_missing_mask = hrrr_forecast_df[
    "precipitation_rate_mm_per_hour"
].isna()
hrrr_forecast_df.to_parquet(
    HRRR_COMBINED_PATH,
    index=False,
)

hrrr_download_summary_df = pd.DataFrame(
    [
        {
            "row_count": len(hrrr_forecast_df),
            "unique_run_count": hrrr_forecast_df[
                "init_time_utc"
            ].nunique(),
            "first_init_time_utc": hrrr_forecast_df[
                "init_time_utc"
            ].min(),
            "last_init_time_utc": hrrr_forecast_df[
                "init_time_utc"
            ].max(),
            "first_valid_time_utc": hrrr_forecast_df[
                "valid_time_utc"
            ].min(),
            "last_valid_time_utc": hrrr_forecast_df[
                "valid_time_utc"
            ].max(),
            "borough_count": hrrr_forecast_df["borough"].nunique(),
            "duplicate_key_count": hrrr_duplicate_key_count,
            "lead_zero_precipitation_missing_rows": int(
                hrrr_precipitation_missing_mask.sum()
            ),
            "combined_file_size_mb": (
                HRRR_COMBINED_PATH.stat().st_size / 1e6
            ),
        }
    ]
)
print(f"Saved HRRR forecasts to: {HRRR_COMBINED_PATH}")
hrrr_download_summary_df

Loading cached partition: hrrr_forecast_init_2025-03.parquet
Loading cached partition: hrrr_forecast_init_2025-04.parquet
Loading cached partition: hrrr_forecast_init_2025-05.parquet
Loading cached partition: hrrr_forecast_init_2025-06.parquet
Loading cached partition: hrrr_forecast_init_2025-07.parquet
Loading cached partition: hrrr_forecast_init_2025-08.parquet
Loading cached partition: hrrr_forecast_init_2025-09.parquet
Loading cached partition: hrrr_forecast_init_2025-10.parquet
Loading cached partition: hrrr_forecast_init_2025-11.parquet
Loading cached partition: hrrr_forecast_init_2025-12.parquet
Loading cached partition: hrrr_forecast_init_2026-01.parquet
Saved HRRR forecasts to: /Users/lavenderle/project-1-individual-LavenderLe7/data/standardized/hrrr_forecasts/hrrr_borough_forecasts_standardized.parquet


,row_count,unique_run_count,first_init_time_utc,last_init_time_utc,first_valid_time_utc,last_valid_time_utc,borough_count,duplicate_key_count,lead_zero_precipitation_missing_rows,combined_file_size_mb
0,270970,1106,2025-03-31 00:00:00+00:00,2026-01-01 06:00:00+00:00,2025-03-31 00:00:00+00:00,2026-01-03 06:00:00+00:00,5,0,5530,2.9502


## 4.3 Holiday reference data

In [86]:
# Load NYC holiday dates for calendar feature construction
HOLIDAY_CALENDAR_PATH = REFERENCE_DATA_ROOT / "holiday_calendar.csv"

holiday_reference_df = pd.read_csv(
    HOLIDAY_CALENDAR_PATH,
    parse_dates=["date"],
)

nyc_holiday_dates = tuple(
    holiday_reference_df.loc[
        holiday_reference_df["applies_to_nyc"].eq(1),
        "date",
    ]
    .dt.strftime("%Y-%m-%d")
    .tolist()
)

# 5. Modelling panel construction

## 5.1 Complete borough-period panel and target aggregation

In [87]:
# Define the complete borough by four-hour modelling grid
period_start_values = pd.date_range(
    start=ANALYSIS_START,
    end=ANALYSIS_END_EXCLUSIVE,
    inclusive="left",
    freq=f"{PERIOD_HOURS}h",
).to_pydatetime()

borough_period_schema = T.StructType(
    [
        T.StructField("borough", T.StringType(), False),
        T.StructField("period_start", T.TimestampNTZType(), False),
    ]
)

borough_period_rows = [
    (borough, period_start)
    for borough in NYC_BOROUGHS
    for period_start in period_start_values
]

complete_borough_period_grid_df = spark.createDataFrame(
    borough_period_rows,
    schema=borough_period_schema,
)

# Report the expected and observed grid dimensions
EXPECTED_PERIOD_COUNT = len(period_start_values)
EXPECTED_PANEL_ROW_COUNT = EXPECTED_PERIOD_COUNT * len(NYC_BOROUGHS)
actual_grid_row_count = complete_borough_period_grid_df.count()

print(
    f"Complete grid: {EXPECTED_PERIOD_COUNT:,} periods × "
    f"{len(NYC_BOROUGHS)} boroughs = {actual_grid_row_count:,} rows"
)

Complete grid: 1,650 periods × 5 boroughs = 8,250 rows


In [88]:
# Reuse Spark expressions across the calendar feature definitions
period_hour = F.hour("period_start")

# Map day of year onto an annual cycle for cyclical seasonality
annual_cycle_angle = (
    2
    * F.lit(np.pi)
    * (F.dayofyear("period_start") - F.lit(1))
    / F.lit(365.25)
)

# Add calendar, trend, holiday, and chronological split features to
# every borough by four-hour period in the complete analysis grid
calendar_grid_df = (
    complete_borough_period_grid_df
    .withColumn(
        "time_block",
        F.format_string(
            "%02d:00-%02d:59",
            period_hour,
            period_hour + F.lit(PERIOD_HOURS - 1),
        ),
    )
    .withColumn(
        "day_of_week",
        F.date_format("period_start", "EEEE"),
    )
    .withColumn(
        "day_of_week_number",
        F.pmod(
            F.dayofweek("period_start") + F.lit(5),
            F.lit(7),
        ).cast("int"),
    )
    .withColumn(
        "day_of_year_sin",
        F.sin(annual_cycle_angle),
    )
    .withColumn(
        "day_of_year_cos",
        F.cos(annual_cycle_angle),
    )
    .withColumn(
        "days_since_analysis_start",
        F.datediff("period_start", F.lit(ANALYSIS_START)),
    )
    .withColumn(
        "is_holiday",
        F.date_format("period_start", "yyyy-MM-dd").isin(
            *nyc_holiday_dates
        ),
    )
    .withColumn(
        "split_label",
        F.when(
            F.col("period_start") < F.lit(TRAIN_END_EXCLUSIVE),
            F.lit("training"),
        )
        .when(
            F.col("period_start") < F.lit(VALIDATION_END_EXCLUSIVE),
            F.lit("validation"),
        )
        .otherwise(F.lit("test")),
    )
)

In [89]:
# Load the standardised trip-level WAV request data
standardized_wav_requests_df = spark.read.parquet(
    *[str(path) for path in STANDARDIZED_WAV_REQUEST_PATHS]
)

# Aggregate trip-level records to borough by four-hour period counts
wav_period_counts_df = (
    standardized_wav_requests_df
    .groupBy(
        F.col("pickup_borough").alias("borough"),
        "period_start",
    )
    .agg(
        F.count("*").alias("wav_request_count"),
        F.sum("late_10").alias("late_trip_count"),
    )
)

In [90]:
# Join WAV counts to the complete grid and construct the late-share target
wav_panel_df = (
    calendar_grid_df
    .join(
        wav_period_counts_df,
        on=["borough", "period_start"],
        how="left",
    )
    .fillna(0, subset=["wav_request_count", "late_trip_count"])
    .withColumn(
        "target_observed",
        F.col("wav_request_count") > 0,
    )
    .withColumn(
        "late_share",
        F.when(
            F.col("target_observed"),
            F.col("late_trip_count") / F.col("wav_request_count"),
        ),
    )
)

## 5.2 Supporting-data integration

In [91]:
# Aggregate borough-level WAV requests to citywide four-hour totals
citywide_wav_counts_df = (
    wav_period_counts_df
    .groupBy("period_start")
    .agg(
        F.sum("wav_request_count").alias(
            "citywide_wav_request_count"
        )
    )
)

# Integrate citywide WAV demand and weather data into the modelling panel
panel_source_df = (
    wav_panel_df
    .join(citywide_wav_counts_df, on="period_start", how="left")
    .join(weather_period_spark_df, on="period_start", how="left")
    .fillna(0, subset=["citywide_wav_request_count"])
)

## 5.3 Mature historical features and curated panel output

Issue each forecast at the target-period start. Construct demand and risk
predictors only from source periods that ended at least four hours earlier,
so the newest eligible completed-trip cohort is lag 2. Preserve observed
LCD fields as descriptive context rather than predictive features.

In [92]:
PERIODS_PER_DAY = 24 // PERIOD_HOURS
PERIODS_PER_WEEK = 7 * PERIODS_PER_DAY

# Define borough-level chronological windows for historical features
BOROUGH_TIME_WINDOW = Window.partitionBy("borough").orderBy(
    "period_start"
)

MATURED_24H_WINDOW = BOROUGH_TIME_WINDOW.rowsBetween(
    -(PERIODS_PER_DAY + DATA_MATURITY_BUFFER_PERIODS),
    -HISTORICAL_FEATURE_LAG_PERIODS,
)

MATURED_7D_WINDOW = BOROUGH_TIME_WINDOW.rowsBetween(
    -(PERIODS_PER_WEEK + DATA_MATURITY_BUFFER_PERIODS),
    -HISTORICAL_FEATURE_LAG_PERIODS,
)

# Define the expanding history window for the same weekday and time block
SEASONAL_HISTORY_WINDOW = (
    Window
    .partitionBy("borough", "day_of_week_number", "time_block")
    .orderBy("period_start")
    .rowsBetween(Window.unboundedPreceding, -1)
)

In [93]:
# Construct demand features with one full period for data maturity
feature_panel_df = (
    panel_source_df
    .withColumn(
        "lag_2_wav_request_count",
        F.lag("wav_request_count", HISTORICAL_FEATURE_LAG_PERIODS)
        .over(BOROUGH_TIME_WINDOW),
    )
    .withColumn(
        "rolling_24h_wav_request_count",
        F.sum("wav_request_count").over(MATURED_24H_WINDOW),
    )
    .withColumn(
        "lag_2_citywide_wav_request_count",
        F.lag(
            "citywide_wav_request_count",
            HISTORICAL_FEATURE_LAG_PERIODS,
        ).over(
            BOROUGH_TIME_WINDOW
        ),
    )
)

# Apply log1p transformation to historical count features
LOG_COUNT_FEATURES = (
    "lag_2_wav_request_count",
    "rolling_24h_wav_request_count",
    "lag_2_citywide_wav_request_count",
)

for feature_name in LOG_COUNT_FEATURES:
    feature_panel_df = feature_panel_df.withColumn(
        f"log1p_{feature_name}",
        F.log1p(F.col(feature_name)),
    )

In [94]:
# Construct matured historical risk features and availability indicators
feature_panel_df = (
    feature_panel_df
    .withColumn(
        "lag_2_late_share",
        F.lag("late_share", HISTORICAL_FEATURE_LAG_PERIODS)
        .over(BOROUGH_TIME_WINDOW),
    )
    .withColumn(
        "previous_24h_late_count",
        F.sum("late_trip_count").over(MATURED_24H_WINDOW),
    )
    .withColumn(
        "previous_24h_request_count",
        F.sum("wav_request_count").over(MATURED_24H_WINDOW),
    )
    .withColumn(
        "previous_7d_late_count",
        F.sum("late_trip_count").over(MATURED_7D_WINDOW),
    )
    .withColumn(
        "previous_7d_request_count",
        F.sum("wav_request_count").over(MATURED_7D_WINDOW),
    )
    .withColumn(
        "rolling_24h_late_share",
        F.when(
            F.col("previous_24h_request_count") > 0,
            F.col("previous_24h_late_count")
            / F.col("previous_24h_request_count"),
        ),
    )
    .withColumn(
        "rolling_7d_late_share",
        F.when(
            F.col("previous_7d_request_count") > 0,
            F.col("previous_7d_late_count")
            / F.col("previous_7d_request_count"),
        ),
    )
    .withColumn(
        "seasonal_history_late_count",
        F.sum("late_trip_count").over(SEASONAL_HISTORY_WINDOW),
    )
    .withColumn(
        "seasonal_history_request_count",
        F.sum("wav_request_count").over(SEASONAL_HISTORY_WINDOW),
    )
    .withColumn(
        "historical_same_slot_late_share",
        F.when(
            F.col("seasonal_history_request_count") > 0,
            F.col("seasonal_history_late_count")
            / F.col("seasonal_history_request_count"),
        ),
    )
    .withColumn(
        "lag_2_late_share_available",
        F.col("lag_2_late_share").isNotNull(),
    )
    .withColumn(
        "rolling_24h_late_share_available",
        F.col("rolling_24h_late_share").isNotNull(),
    )
    .withColumn(
        "rolling_7d_late_share_available",
        F.col("rolling_7d_late_share").isNotNull(),
    )
    .withColumn(
        "same_slot_late_share_available",
        F.col("historical_same_slot_late_share").isNotNull(),
    )
)

In [95]:
# Aggregate development-period request support and outcomes by borough
wav_borough_period_support_df = (
    feature_panel_df
    .filter(
        F.col("period_start") < F.lit(VALIDATION_END_EXCLUSIVE)
    )
    .groupBy("borough")
    .agg(
        F.count("*").alias("period_count"),
        F.sum(F.col("target_observed").cast("long")).alias(
            "target_observed_period_count"
        ),
        F.sum("wav_request_count").alias("wav_request_count"),
        F.sum("late_trip_count").alias("late_trip_count"),
        F.percentile_approx(
            "wav_request_count",
            [0.25, 0.50, 0.75],
            10_000,
        ).alias("request_count_quantiles"),
    )
    .orderBy("borough")
    .toPandas()
)
# Unpack borough request-count quartiles into readable columns
support_quantiles_df = pd.DataFrame(
    wav_borough_period_support_df.pop("request_count_quantiles").tolist(),
    columns=[
        "p25_request_count",
        "median_request_count",
        "p75_request_count",
    ],
)
wav_borough_period_support_df = pd.concat(
    [wav_borough_period_support_df, support_quantiles_df],
    axis=1,
)
wav_borough_period_support_df["zero_request_period_count"] = (
    wav_borough_period_support_df["period_count"]
    - wav_borough_period_support_df["target_observed_period_count"]
)
wav_borough_period_support_df["late_share"] = (
    wav_borough_period_support_df["late_trip_count"]
    / wav_borough_period_support_df["wav_request_count"]
)
wav_borough_period_support_df[
    [
        "borough",
        "wav_request_count",
        "late_share",
        "median_request_count",
        "zero_request_period_count",
    ]
]

,borough,wav_request_count,late_share,median_request_count,zero_request_period_count
0,Bronx,65949,0.1111,35,0
1,Brooklyn,122612,0.0786,81,0
2,Manhattan,152228,0.0825,91,0
3,Queens,80563,0.1201,57,0
4,Staten Island,7281,0.1603,4,248


In [96]:
# Freeze the mature service-history fields retained in the panel
CURATED_HISTORY_FEATURES = (
    "borough",
    "time_block",
    "day_of_week",
    "is_holiday",
    "day_of_year_sin",
    "day_of_year_cos",
    "days_since_analysis_start",
    *(f"log1p_{name}" for name in LOG_COUNT_FEATURES),
    "lag_2_late_share",
    "rolling_24h_late_share",
    "rolling_7d_late_share",
    "historical_same_slot_late_share",
    "lag_2_late_share_available",
    "rolling_24h_late_share_available",
    "rolling_7d_late_share_available",
    "same_slot_late_share_available",
)

In [97]:
# Retain targets, descriptive context, and all contracted predictors
CURATED_PANEL_COLUMNS = (
    "period_start",
    "borough",
    "time_block",
    "day_of_week",
    "day_of_week_number",
    "day_of_year_sin",
    "day_of_year_cos",
    "days_since_analysis_start",
    "is_holiday",
    "split_label",
    "wav_request_count",
    "late_trip_count",
    "late_share",
    "target_observed",
    "citywide_wav_request_count",
    "observed_mean_temperature_c",
    "observed_max_wind_speed_ms",
    "observed_min_visibility_km",
    "observed_any_precipitation",
    "observed_any_rain",
    "observed_any_snow",
    "weather_coverage_share",
    *LOG_COUNT_FEATURES,
    *CURATED_HISTORY_FEATURES,
)

# Remove repeated fields while preserving the declared column order
CURATED_PANEL_COLUMNS = tuple(dict.fromkeys(CURATED_PANEL_COLUMNS))

In [98]:
# Materialise the ordered Spark panel as the reusable modelling dataset
curated_wav_panel_df = (
    feature_panel_df
    .select(*CURATED_PANEL_COLUMNS)
    .orderBy("period_start", "borough")
    .toPandas()
)
curated_wav_panel_df.to_parquet(
    CURATED_WAV_PANEL_PATH,
    index=False,
    compression="zstd",
)

# Reload the saved artifact so later sections use the reproducible output
saved_curated_wav_panel_df = pd.read_parquet(CURATED_WAV_PANEL_PATH)

print(
    f"Saved curated panel: {len(saved_curated_wav_panel_df):,} rows × "
    f"{len(saved_curated_wav_panel_df.columns):,} columns"
)
print(f"Curated panel path: {CURATED_WAV_PANEL_PATH}")

Saved curated panel: 8,250 rows × 36 columns
Curated panel path: /Users/lavenderle/project-1-individual-LavenderLe7/data/curated/wav_service_risk/borough_four_hour_panel.parquet


In [99]:
# Terminate the local Spark session to release system resources
spark.stop()
print("Spark session successfully terminated.")
print(f"Curated panel written to: {CURATED_WAV_PANEL_PATH}")

Spark session successfully terminated.
Curated panel written to: /Users/lavenderle/project-1-individual-LavenderLe7/data/curated/wav_service_risk/borough_four_hour_panel.parquet
